# ADAPT-QAOA

All twenty source stages, eight decision qubits, a three-layer ordinary comparator, all 74 candidate mixers in the original order, and adaptive growth until the source scaled gradient norm falls below 0.027. Each optimization allows 60 objective evaluations.

Native COBYLA replaces the hosted optimizer with the source ramp, an explicit query cap, and exact state expectations. Independent final draws use 1000 shots per variant with fixed local seeds. Public post-synthesis gate conversion measures the original CZ/RZ/RX/RY counts after verifying absolute amplitudes; the logical synthesis path preserves phase. A local 24-layer guard raises an error instead of treating exhaustion as convergence. This run does not reproduce historical hosted sampling or optimizer trajectories.

## Use the native optimizer and plotting tools

In [ ]:
from pathlib import Path

from IPython.display import Image, display

from examples.synthesis.applications.optimization.adapt_qaoa.overview.model import (
    costs,
    pool,
)
from examples.synthesis.applications.optimization.adapt_qaoa.overview.plots import (
    plot_cost,
)
from examples.synthesis.applications.optimization.adapt_qaoa.overview.quantum import (
    build_program,
)
from examples.synthesis.applications.optimization.adapt_qaoa.overview.resources import (
    SOURCE_BASIS,
    source_basis_metrics,
)
from examples.synthesis.applications.optimization.adapt_qaoa.overview.workflow import (
    BUDGET,
    THRESHOLD,
    run_workflow,
)

directory = Path("artifacts/development/adapt-qaoa")


## Build the original eight-spin energy table

In [ ]:
diagonal = costs()
print("Energy entries and exact minimum:", len(diagonal), float(diagonal.min()))


## Describe the cost operator

In [ ]:
print(
    "The cost phase applies the same eight local Z and sixteen pairwise ZZ coefficients."
)


## Describe the ordinary mixer

In [ ]:
print("Ordinary QAOA rotates all eight qubits about X.")


## Show the initial schedule

In [ ]:
print("Ordinary initial gammas: 0, 0.5, 1; betas: 1, 0.5, 0.")


## Specialize the three-layer comparison

In [ ]:
mixers = pool()
starting = build_program((mixers[0],) * 3, (0.0, 0.5, 1.0, 1.0, 0.5, 0.0))


## Check the original target gate basis

In [ ]:
print("Target gates:", SOURCE_BASIS)
starting_metrics = source_basis_metrics(starting)


## Show width, depth and operation counts

In [ ]:
print(starting_metrics)


## Use the full expected cost at each query

In [ ]:
print("Exact mean energy over all 256 states.")


## Optimize both circuits under the source budgets

In [ ]:
result = run_workflow()
print("Ordinary objective queries:", len(result.ordinary.trace), "/", BUDGET)


## Plot actual ordinary objective values

In [ ]:
display(
    Image(
        filename=str(
            plot_cost(result.ordinary, "Ordinary QAOA", directory / "ordinary.png")
        )
    )
)


## Print the best observed ordinary words

In [ ]:
print("Ordinary final parameters:", result.ordinary.parameters)
for ordinary_row in result.ordinary_samples[:10]:
    print(ordinary_row)


## Retain the complete ordered mixer pool

In [ ]:
print("Mixer pool size:", len(mixers))
assert len(mixers) == 74


## Describe the scaled commutator stop rule

In [ ]:
print(
    "Stop when the L2 norm of the 74 absolute commutators divided by 74 is below",
    THRESHOLD,
)


## Inspect every adaptive iteration

In [ ]:
for layer in result.layers:
    print(layer.mixers, len(layer.optimization.trace), layer.gradient_norm)
assert result.layers[-1].gradient_norm < THRESHOLD


## Read the finite final adaptive sample

In [ ]:
print("Adaptive final draws:", sum(row.count for row in result.adaptive_samples))


## Plot the final adaptive objective

In [ ]:
final = result.layers[-1].optimization
display(
    Image(filename=str(plot_cost(final, "Adaptive QAOA", directory / "adaptive.png")))
)


## Read the complete adaptive sample table

In [ ]:
print(result.adaptive_samples)


## Show ten lowest observed energies

In [ ]:
print("Adaptive final parameters:", final.parameters)
for adaptive_row in result.adaptive_samples[:10]:
    print(adaptive_row)


## Sort observed words by frequency

In [ ]:
print(sorted(result.adaptive_samples, key=lambda row: (-row.count, row.word)))
